In [1]:
# ¿Qué cambios de un extracto tributario permiten o impiden analizar ingresos por código postal?
#
# lote nuevo ──► contrato ──► ACCEPT / REVIEW / REJECT ──► análisis

from pathlib import Path

import pandas as pd

ROOT = next(
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "data").is_dir() and (path / "submission").is_dir()
)
SOURCE = ROOT / "data" / "vermont.csv"
OUTPUT = ROOT / "submission" / "contract_report.csv"


In [2]:
# La fuente usa una fila por código postal y tramo de ingreso.

tax = pd.read_csv(SOURCE)
tax.shape, tax[["zipcode", "agi_stub", "N1", "A00100"]].head()


((1476, 147),
    zipcode  agi_stub      N1   A00100
 0        0         1  111580  1314871
 1        0         2   82760  3006587
 2        0         3   46270  2851399
 3        0         4   30070  2617891
 4        0         5   39530  5472982)

In [3]:
# El contrato solo declara lo que el análisis necesita, no las 147 columnas de la fuente.

REQUIRED = {"STATEFIPS", "STATE", "zipcode", "agi_stub", "N1", "A00100"}


def validate(frame):
    if not REQUIRED.issubset(frame.columns):
        return "FAIL", "BREAKING", "REJECT"
    if frame["STATE"].ne("VT").any() or frame["STATEFIPS"].ne(50).any():
        return "FAIL", "BREAKING", "REJECT"
    if not frame["agi_stub"].between(1, 6).all() or frame["N1"].lt(0).any():
        return "FAIL", "BREAKING", "REJECT"
    if frame[["zipcode", "agi_stub"]].duplicated().any():
        return "FAIL", "BREAKING", "REJECT"
    if frame["zipcode"].eq(0).any():
        return "REVIEW", "SCOPE", "FILTER_ZIPCODE_0"
    change = "COMPATIBLE" if set(frame.columns) - set(tax.columns) else "NONE"
    return "PASS", change, "ACCEPT"


In [4]:
# Los agregados estatales son válidos, pero no responden preguntas por código postal.

cases = {
    "valid_zipcode_scope": tax.loc[tax["zipcode"].ne(0)].copy(),
    "statewide_total_present": tax,
    "missing_income_group": tax.drop(columns="agi_stub"),
    "wrong_state": tax.assign(STATE="XX"),
    "invalid_income_group": tax.assign(agi_stub=7),
    "optional_column": tax.loc[tax["zipcode"].ne(0)].assign(source_release="2017"),
}
report = pd.DataFrame(
    [(name, *validate(frame)) for name, frame in cases.items()],
    columns=["batch_name", "status", "change_type", "action"],
)
report.to_csv(OUTPUT, index=False)
report


,batch_name,status,change_type,action
0,valid_zipcode_scope,PASS,NONE,ACCEPT
1,statewide_total_present,REVIEW,SCOPE,FILTER_ZIPCODE_0
2,missing_income_group,FAIL,BREAKING,REJECT
3,wrong_state,FAIL,BREAKING,REJECT
4,invalid_income_group,FAIL,BREAKING,REJECT
5,optional_column,PASS,COMPATIBLE,ACCEPT
